In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.auxiliars_for_modeling import apply_cyclical_encoding

import dtale
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi 
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance 


from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
import matplotlib.pyplot as plt

import gc



load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)
#the setup (we encapsulated that in a function for keep it constante during the joining to analize purely the gains of each table)
cv , hiperparams = get_baseline_setup() 





In [ ]:

#lets try with the main table without any treatment in the data.
application_train_df = pd.read_csv(cfg.RAW_DATA_DIR / "application_train.csv")

#minimun preparations necessary to be able to train the model with application_train
Y= application_train_df["TARGET"]
X= application_train_df.drop(columns=["TARGET"])
X.drop(columns=["SK_ID_CURR"],inplace=True)
X= cast_object_into_categoricals(X)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline")
#0.744 OOF auc is our baseline.

#cleaning memory
del application_train_df,X,Y  
gc.collect()

In [ ]:
#now let's repeat the set up with the version of the silver layer (Cleaned application_train)
#Therefore, this part need execute make_dataset first to generate the fold 01_cleaned

cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
X,Y = prepare_columns(cleaned_application_train)
X= cast_object_into_categoricals(X)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline_cleaned_data")
#0.745 OOF auc. Just cleaning the data give us +0.1%, and with less risk of overfitting.


#cleaning memory
del cleaned_application_train,X,Y  
gc.collect()

In [ ]:
cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
dtale.show(cleaned_application_train[:100])

In [9]:
cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")

cleaned_application_train["ratio_debt_income"] = (cleaned_application_train["amt_credit"] /  (cleaned_application_train["amt_income_total"]))

cleaned_application_train["ratio_debt_age"]= cleaned_application_train["amt_credit"] /(cleaned_application_train["days_birth"] * -1) 
cleaned_application_train["ratio_days_employed_days_lived"]= cleaned_application_train["days_employed"] /(cleaned_application_train["days_birth"] * -1) 
cleaned_application_train["kui_ratio"] =  np.where(cleaned_application_train["days_employed"] != 0, cleaned_application_train["amt_credit"] / ((cleaned_application_train["days_employed"] * -1) * cleaned_application_train["amt_income_total"]), 0)
cleaned_application_train["ratio_good_credit"]= cleaned_application_train["amt_goods_price"] / cleaned_application_train["amt_credit"]
cleaned_application_train["ratio_annuity_income"] = cleaned_application_train["amt_annuity"] / cleaned_application_train["amt_income_total"]
#cleaned_application_train["toxic_feature_1"] = cleaned_application_train["cnt_children"] / cleaned_application_train["amt_income_total"]

cleaned_application_train["credit_duration"]= cleaned_application_train["amt_credit"] / cleaned_application_train["amt_annuity"]
cleaned_application_train["ext_1_x_2"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_2"]
cleaned_application_train["ext_2_x_3"] = cleaned_application_train["ext_source_2"] * cleaned_application_train["ext_source_3"]
cleaned_application_train["ext_1_x_3"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_3"]







cleaned_application_train = pd.get_dummies(cleaned_application_train,columns=["organization_type"])
cleaned_application_train = pd.get_dummies(cleaned_application_train,columns=["education_type"])
#cleaned_application_train = pd.get_dummies(cleaned_application_train,columns=["occupation_type"])


#cleaned_application_train["amount_of_scores_in_missing"] = cleaned_application_train["ext_source_1_is_missing"] + cleaned_application_train["ext_source_2_is_missing"] + cleaned_application_train["ext_source_3_is_missing"]
ext_cols = ['ext_source_1', 'ext_source_2', 'ext_source_3']

building_features_names = [
    "APARTMENTS_AVG","BASEMENTAREA_AVG","YEARS_BEGINEXPLUATATION_AVG","YEARS_BUILD_AVG",
    "COMMONAREA_AVG","ELEVATORS_AVG","ENTRANCES_AVG","FLOORSMAX_AVG","FLOORSMIN_AVG",
    "LANDAREA_AVG","LIVINGAPARTMENTS_AVG","LIVINGAREA_AVG","NONLIVINGAPARTMENTS_AVG",
    "NONLIVINGAREA_AVG","APARTMENTS_MODE","BASEMENTAREA_MODE","YEARS_BEGINEXPLUATATION_MODE",
    "YEARS_BUILD_MODE","COMMONAREA_MODE","ELEVATORS_MODE","ENTRANCES_MODE","FLOORSMAX_MODE",
    "FLOORSMIN_MODE","LANDAREA_MODE","LIVINGAPARTMENTS_MODE","LIVINGAREA_MODE",
    "NONLIVINGAPARTMENTS_MODE","NONLIVINGAREA_MODE","APARTMENTS_MEDI","BASEMENTAREA_MEDI",
    "YEARS_BEGINEXPLUATATION_MEDI","YEARS_BUILD_MEDI","COMMONAREA_MEDI","ELEVATORS_MEDI",
    "ENTRANCES_MEDI","FLOORSMAX_MEDI","FLOORSMIN_MEDI","LANDAREA_MEDI","LIVINGAPARTMENTS_MEDI",
    "LIVINGAREA_MEDI","NONLIVINGAPARTMENTS_MEDI","NONLIVINGAREA_MEDI","FONDKAPREMONT_MODE",
    "HOUSETYPE_MODE","TOTALAREA_MODE","WALLSMATERIAL_MODE","EMERGENCYSTATE_MODE"
    ]

building_features_names = [col.lower() for col in building_features_names]

categorical_bldg = ['fondkapremont_mode', 'housetype_mode', 'wallsmaterial_mode', 'emergencystate_mode']

numeric_bldg = [col for col in building_features_names if col not in categorical_bldg]



# Agregaciones horizontales (axis=1)
cleaned_application_train["ext_source_mean"] = cleaned_application_train[ext_cols].mean(axis=1)
#cleaned_application_train["ext_source_max"] = cleaned_application_train[ext_cols].max(axis=1)
#cleaned_application_train["ext_source_min"] = cleaned_application_train[ext_cols].min(axis=1)
cleaned_application_train["ext_source_std"] = cleaned_application_train[ext_cols].std(axis=1)



#cleaned_application_train["debt_age_x_ext_source"] = np.where(  cleaned_application_train["ext_source_mean"] != 0,  cleaned_application_train["ratio_debt_age"] / cleaned_application_train["ext_source_mean"], 0)


# Agregaciones horizontales (axis=1)
cleaned_application_train["building_score_mean"] = cleaned_application_train[numeric_bldg].mean(axis=1)
cleaned_application_train["building_score_max"] = cleaned_application_train[numeric_bldg].max(axis=1)
cleaned_application_train["building_score_min"] = cleaned_application_train[numeric_bldg].min(axis=1)
cleaned_application_train["building_score_std"] = cleaned_application_train[numeric_bldg].std(axis=1)
cleaned_application_train["building_score_sum"] = cleaned_application_train[numeric_bldg].sum(axis=1)

cleaned_application_train["building_features_nan_count"] = cleaned_application_train[building_features_names].isnull().sum(axis=1)

#numeric_bldg.append("obs_30_cnt_social_circle")
cleaned_application_train= cleaned_application_train.drop(columns=numeric_bldg)
#cleaned_application_train= cleaned_application_train.drop(columns=["flag_city_not_work"] )


X,Y = prepare_columns(cleaned_application_train)

X= cast_object_into_categoricals(X)

pfi= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_baseline_cleaned_data.csv")

X= clean_importance_zero_and_negative_pfi(pfi,X)


#cleaned_application_train.to_parquet(cfg.PROCESSED_DIR / "pruned_baseline.parquet")

model= xgb.XGBClassifier(**hiperparams)

run_cv_tracked_mlflow(model,hiperparams,cv,X,Y,experiment_name,"baseline")
#run_cv_tracked_mlflow_f_i(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline_cleaned_data",run_pfi=True)
#0.753 OOF auc with basic feature engineering


#cleaning memory
#del cleaned_application_train,X,Y  
#gc.collect()

eliminando ['flag_city_not_work', 'flag_phone', 'cnt_family_members', 'flag_live_city_not_work', 'flag_email', 'organization_type_Trade: type 3', 'organization_type_Other', 'housetype_mode', 'organization_type_Business Entity Type 1', 'organization_type_University', 'flag_region_not_live', 'organization_type_Bank', 'flag_live_region_not_work', 'education_type_Lower secondary', 'flag_own_car', 'organization_type_Industry: type 7', 'organization_type_Trade: type 2', 'organization_type_Security', 'emergencystate_mode', 'organization_type_Industry: type 1', 'organization_type_Industry: type 3', 'flag_document_18', 'organization_type_Industry: type 5', 'flag_document_11', 'organization_type_Transport: type 2', 'organization_type_Restaurant', 'organization_type_Other trade', 'flag_document_16', 'ext_source_3_is_missing', 'organization_type_Trade: type 6', 'organization_type_Telecom', 'organization_type_XNA', 'amt_goods_price_is_missing', 'client_without_querys', 'organization_type_Industry: 

(0.7622191450035185, 0.0028281379450524444)

In [36]:
application_train_proccesed= pd.read_parquet(cfg.PROCESSED_DIR / "application_train_for_experiments")

application_train_proccesed = pd.get_dummies(application_train_proccesed,columns=["education_type"])


application_train_proccesed = apply_cyclical_encoding(application_train_proccesed,"hour_apply_start",24)

application_train_proccesed

dict_to_map_week_days = {
        "MONDAY" : 1,
        "TUESDAY" : 2,
        "THURSDAY": 3,
        "WEDNESDAY" : 4,
        "FRIDAY" : 5,
        "SATURDAY" : 6,
        "SUNDAY" : 7
    }


application_train_proccesed["weekday_appr_process_start"] =application_train_proccesed["weekday_appr_process_start"].map(dict_to_map_week_days) 

apply_cyclical_encoding(application_train_proccesed,"weekday_appr_process_start",7)

application_train_proccesed= application_train_proccesed.drop(columns=["weekday_appr_process_start","hour_apply_start"])

categorical_features= ["organization_type","occupation_type"] #"name_type_suite",,"weekday_appr_process_start"

X,Y = prepare_columns(application_train_proccesed)

model=xgb.XGBClassifier(**hiperparams)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'target_encode_cat', 
            TargetEncoder(
                categories='auto',      
                smooth= 50,          
                cv=5,                   
                random_state=42
            ), 
            categorical_features
        )
    ],
    remainder='passthrough' 
)

preprocessor.set_output(transform="pandas") #god bless this

# 4. Construimos la Pipeline final
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', model)
])

X= cast_object_into_categoricals(X)

#pfi= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_pipeline_pruned.csv")

#X= clean_importance_zero_and_negative_pfi(pfi,X,0.000148)



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"pipeline_pruned")

application_train_proccesed.to_parquet(cfg.PROCESSED_DIR / "target_baseline")


🏃 View run pipeline_pruned_child_1 at: http://localhost:5000/#/experiments/3/runs/3cd7c422dfae4b4d9f9251655cd5e363
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run pipeline_pruned_child_2 at: http://localhost:5000/#/experiments/3/runs/0d09369630a94b7aa1a65bd74b909b66
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run pipeline_pruned_child_3 at: http://localhost:5000/#/experiments/3/runs/23846c6181304407942080019eeb8572
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run pipeline_pruned_child_4 at: http://localhost:5000/#/experiments/3/runs/e6e0f42e453b4ef0b6c74da49b2c1d3f
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run pipeline_pruned_child_5 at: http://localhost:5000/#/experiments/3/runs/5370ea503ff1413e9daa4c5dad227122
🧪 View experiment at: http://localhost:5000/#/experiments/3
AUC per fold= 0.761 ± 0.004(std), auc_score_OOF= 0.761 result of CV with 5 folds. 
🏃 View run Parent_pipeline_pruned at: http

In [9]:
application_train_proccesed= pd.read_parquet(cfg.PROCESSED_DIR / "application_train_for_experiments")

categorical_features= ["organization_type","occupation_type","name_income_type"] #"name_type_suite",,"weekday_appr_process_start"

application_train_proccesed = apply_cyclical_encoding(application_train_proccesed,"hour_apply_start",24)

application_train_proccesed

dict_to_map_week_days = {
        "MONDAY" : 1,
        "TUESDAY" : 2,
        "WEDNESDAY" : 3,
        "THURSDAY": 4,
        "FRIDAY" : 5,
        "SATURDAY" : 6,
        "SUNDAY" : 7
    }

application_train_proccesed["weekday_appr_process_start"] =application_train_proccesed["weekday_appr_process_start"].map(dict_to_map_week_days) 

apply_cyclical_encoding(application_train_proccesed,"weekday_appr_process_start",7)

application_train_proccesed= application_train_proccesed.drop(columns=["weekday_appr_process_start","hour_apply_start"])



X,Y = prepare_columns(application_train_proccesed)

model=xgb.XGBClassifier(**hiperparams)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'target_encode_cat', 
            TargetEncoder(
                categories='auto',      
                smooth=50,          
                cv=5,                   
                random_state=42
            ), 
            categorical_features
        )
    ],
    remainder='passthrough' 
)

preprocessor.set_output(transform="pandas") #god bless this

# 4. Construimos la Pipeline final
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', model)
])

X= cast_object_into_categoricals(X)

features_df= pd.read_csv(cfg.ARTIFACTS_DIR / "36.csv")

feature_list=  features_df["feature_name"].to_list()

feature_list = feature_list + ['def_60_cnt_social_circle',"building_score_sum","amt_income_total","hour_apply_start_sin","hour_apply_start_cos","weekday_appr_process_start_sin","weekday_appr_process_start_cos","name_income_type"]


X= X[feature_list] 

X= pd.get_dummies(X,columns=["education_type"])
#X= pd.get_dummies(X,columns=["name_income_type"])



run_cv_tracked_mlflow(pipeline,hiperparams,cv,X,Y,experiment_name,"baseline_white_list")

application_train_proccesed.to_parquet(cfg.PROCESSED_DIR / "main_target.parquet")



🏃 View run baseline_white_list_child_1 at: http://localhost:5000/#/experiments/3/runs/d2e56f3fc39e465ab7a8ad148f60076b
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run baseline_white_list_child_2 at: http://localhost:5000/#/experiments/3/runs/c5efae7e35a14b84800ec72957ba106b
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run baseline_white_list_child_3 at: http://localhost:5000/#/experiments/3/runs/f1755fbdd8da4147ae1219937769b677
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run baseline_white_list_child_4 at: http://localhost:5000/#/experiments/3/runs/c6a4969c9aff439887b05c083ac83bf7
🧪 View experiment at: http://localhost:5000/#/experiments/3
🏃 View run baseline_white_list_child_5 at: http://localhost:5000/#/experiments/3/runs/6fc3e7c546b345308aa0fa9f7c56f880
🧪 View experiment at: http://localhost:5000/#/experiments/3
AUC per fold= 0.764 ± 0.003(std), auc_score_OOF= 0.763 result of CV with 5 folds. 
🏃 View run Parent_base

In [19]:
application_train_proccesed = apply_cyclical_encoding(application_train_proccesed,"hour_apply_start",24)
application_train_proccesed.head()

,id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,ext_source_mean,ext_source_std,building_score_mean,building_score_max,building_score_min,building_score_std,building_score_sum,building_features_nan_count,hour_apply_start_sin,hour_apply_start_cos
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.161787,0.092026,0.141953,0.9722,0.0,0.277737,6.1040,0,0.500000,-0.866025
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.466757,0.219895,0.202974,0.9851,0.0,0.298725,8.7279,0,0.258819,-0.965926
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.642739,0.122792,NaN,NaN,NaN,NaN,0.0000,47,0.707107,-0.707107
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.650442,NaN,NaN,NaN,NaN,NaN,0.0000,47,-0.965926,-0.258819
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.322738,NaN,NaN,NaN,NaN,NaN,0.0000,47,0.258819,-0.965926
